In [ ]:
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install -q datasets
!pip install trl
!huggingface-cli login
!pip install rouge_score evaluate

In [ ]:
# Import necessary packages for the fine-tuning process and finetuning dataset preprocessing
from transformers import AutoModelForCausalLM, AutoTokenizer, EarlyStoppingCallback
import evaluate
from peft import LoraConfig, get_peft_model
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
import nltk
nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize
from datasets import DatasetDict, concatenate_datasets
import os                          # Operating system functionalities
import torch                       # PyTorch library for deep learning
from datasets import load_dataset  # Loading datasets for training
from transformers import (
    AutoModelForCausalLM,          # AutoModel for language modeling tasks
    AutoTokenizer,                # AutoTokenizer for tokenization
    BitsAndBytesConfig,           # Configuration for BitsAndBytes
    HfArgumentParser,             # Argument parser for Hugging Face models
    TrainingArguments,            # Training arguments for model training
    pipeline,                     # Creating pipelines for model inference
    logging,                      # Logging information during training
)
from peft import LoraConfig, PeftModel  # Packages for parameter-efficient fine-tuning (PEFT)
from trl import SFTTrainer         # SFTTrainer for supervised fine-tuning

In [ ]:
# Prepend both the CUDA toolkit path and the system's GPU-driver path:
os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/cuda/lib64:/usr/lib/x86_64-linux-gnu:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

### As we used the same script for finetuning both RuterNorway/Llama-2-7b-chat-norwegian and Normistral-7b-warm, we simply added a new cell for downloading the normistral-7b-warm from huggingface.

In [ ]:
# Downloading the normistral-7b-warm model from hugging face
model_name = "norallm/normistral-7b-warm"

# The summarization dataset
dataset_name = "SamiaT/NorSumm"

# Fine-tuned model name
new_model = "norallm/normistral-7b-warm-norsumm"

In [ ]:
# Downloading the llama-2-7b-chat-norwegian huggingface
model_name = "RuterNorway/Llama-2-7b-chat-norwegian"

# The summarization dataset
dataset_name = "SamiaT/NorSumm"

# Fine-tuned model name
new_model = "Llama-2-7b-chat-norwegian-sum"

In [ ]:
# LoRA attention dimension
lora_r = 32

# Alpha parameter for LoRA scaling
lora_alpha = 64

# Dropout probability for LoRA layers
lora_dropout = 0.05 # QLoRA paper recommends LoRA dropout = 0.05 for small models (7B, 13B)

In [ ]:
################################################################################
# bitsandbytes parameters
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

In [ ]:
################################################################################
# TrainingArguments parameters
################################################################################
callbacks = [EarlyStoppingCallback(early_stopping_patience=1)]

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 4

# Enable fp16/bf16 training
fp16 = False
bf16 = True

# Saving strategy
save_strategy = "epoch"


# Batch size per GPU for training
per_device_train_batch_size = 4

# Batch size per GPU for evaluation
per_device_eval_batch_size = 4

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 2

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 1e-5

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer
optim = "paged_adamw_8bit"

# Learning rate schedule
lr_scheduler_type = "constant"

# Ratio of steps for a linear warmup
warmup_ratio = 0.03

# Saves memory and speeds up training considerably
group_by_length = True


save_strategy="epoch"

# Log every X updates steps
logging_steps = 4

In [ ]:
################################################################################
# SFT parameters
################################################################################

# Load the entire model on the GPU 0
device_map = {"": 0}

In [ ]:
# Loading NorSumm dataset
dataset_nb = load_dataset("samiat/norSumm", "nb")
dataset_nn = load_dataset("samiat/norSumm", "nn")
# Checking the splits of the dataset
print(f"Bokmål dataset: {dataset_nb}, Nynorsk: {dataset_nn}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.45k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/106k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/142k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/30 [00:00<?, ? examples/s]

...

validation-00000-of-00001.parquet:   0%|          | 0.00/106k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/143k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/30 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/33 [00:00<?, ? examples/s]

Bokmål dataset: DatasetDict({
    validation: Dataset({
        features: ['id', 'article', 'summaries'],
        num_rows: 30
    })
    test: Dataset({
        features: ['id', 'article', 'summaries'],
        num_rows: 33
    })
}), Nynorsk: DatasetDict({
    validation: Dataset({
        features: ['id', 'article', 'summaries'],
        num_rows: 30
    })
    test: Dataset({
        features: ['id', 'article', 'summaries'],
        num_rows: 33
    })
})


In [ ]:
def show_samples(dataset, num_samples=3, seed=42):
    sample = dataset["validation"].shuffle(seed=seed).select(range(num_samples))
    for example in sample:
        print(f"\n'>> Article: {example['article']}'")
        print(f"'>> Summaries: {example['summaries']}'")


show_samples(dataset_nb) # Just an example


'>> Article: Svak tenkning | Gavepakke. Hektisk. 2. akt. Elementært. Det store statskirkespørsmålet er ikke hva slags kristendom vi skal ha, men hva slags stat Norge skal være. Kirkedebatten er i ferd med å bli underlig. Helt frivillig går Kirken inn for å redusere egne privilegier. Kirkens egne råd og utvalg sier at det er problematisk at moderne samfunn blander religion og stat på politisk toppnivå. Statskirken er ikke bare en uskyldig anakronisme, men et demokratisk dilemma. Ordningen bør revideres. Med slik argumentasjonslinjen inviterer både Gjønnes-utvalget og Kirkemøtet til prinsipiell politisk debatt om teokrati, demokrati, likestilling og religionsfrihet. Det er profesjonelt gjort, og ville under normale forhold blitt oppfattet som en gavepakke til norske politikere. Men hva skjer? Det viser seg at ledende norske politikere - og LO-folk, av alle - i stedet vil diskutere teologi. Trond Giske, Martin Kolberg og Gerd-Liv Valla har klart å sabotere den politiske debatten om hva s

In [ ]:
# Merge each split of the HF DatasetDicts
combined_df = DatasetDict()
for split in dataset_nn.keys():
    combined_df[split] = concatenate_datasets(
        [dataset_nn[split], dataset_nb[split]]
    )
    combined_df[split] = combined_df[split].shuffle(seed=42)

# Peek at a few examples
show_samples(combined_df)


'>> Article: Tynnslitte nerver | Frp og Høyre. Halvorsen. Visjon og virkelighet. Sponheim. Skiftende meningsmålinger bringer nervene i høyspenn i alle partier i valgkampinnspurten. Sjelden har så mye stått på spill for så mange partier tre dager før et valg. Høyres satsing på to regjeringsalternativer har vært nødvendig, men er samtidig et risikofylt dobbeltspill. Høyre fikk panikk da bildet av Erna Solberg sammen med Dagfinn Høybråten og Lars Sponheim i VG ble tolket som et tegn på at partiet hadde valgt bort Frp til fordel for en gjenopplivning av Bondevik-alliansen. Det passet dårlig at et slikt inntrykk festet seg før velgerne hadde sagt sitt. I Fremskrittspartiet sprer de nå historier om hvor mye Høyre satset for å få tatt et bilde av Solberg sammen med Siv Jensen for balansens skyld. Men da bildet endelig ble tatt i Stortingets vandrehall, benyttet Jensen anledningen til å spille forurettet på en måte som antagelig skadet Frp mer enn Høyre. Da de to møttes i NRK i går kveld, var

In [ ]:
split = combined_df["validation"].train_test_split(test_size=0.2, seed=42)
train_ds = split["train"]       # ~80% of those ~60 examples
val_ds   = split["test"]        # ~20% of those ~60 examples

# 3) Keep your combined test as-is
test_ds  = combined_df["test"]

# 4) Assemble your final 3‑way DatasetDict
full_ds = DatasetDict({
    "train":      train_ds,
    "validation": val_ds,
    "test":       test_ds,
})

print({k: v.num_rows for k, v in full_ds.items()})

{'train': 48, 'validation': 12, 'test': 66}


In [ ]:
exploded = {}
for split, ds in full_ds.items():
    # pandas DataFrame
    df = ds.to_pandas().reset_index(drop=True)

    # explode the 'summaries' list so each row has one summary string
    df = df.explode("summaries").reset_index(drop=True)

    # convert back to a huggingface Dataset
    exploded[split] = Dataset.from_pandas(df, preserve_index=False)

# reassemble as a DatasetDict
exploded = DatasetDict(exploded)

print({k: v.num_rows for k, v in exploded.items()})

{'train': 144, 'validation': 36, 'test': 198}


In [ ]:
# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

In [ ]:
# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

Your GPU supports bfloat16: accelerate training with bf16=True


In [ ]:
# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map
)
model.config.use_cache = False
model.config.pretraining_tp = 1

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
# See how many trainable parameters and percentage

print_trainable_parameters(model)

trainable params: 262410240 || all params: 3500412928 || trainable%: 7.496550989769399


In [ ]:
# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
max_input_length  = 512
max_target_length = 120

def preprocess_function(examples):
    # make everything to plain Python str
    articles  = [str(a) for a in examples["article"]]
    summaries = [str(s) for s in examples["summaries"]]


    # tokenize inputs and targets
    model_inputs = tokenizer(
        articles,
        max_length=max_input_length,
        truncation=True,
    )
    labels = tokenizer(
        summaries,
        max_length=max_target_length,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized = exploded.map(preprocess_function, batched=True)

Map:   0%|          | 0/144 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

In [ ]:
# Define train, evaluation and test dataset
train_dataset = tokenized["train"]
eval_dataset  = tokenized["validation"]
test_dataset  = tokenized["test"]

In [ ]:
def postprocess_text(preds, labels):
    """
    Postprocesses the predictions and labels to remove extra whitespace and newlines.
    """
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    preds = ["\n".join(sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(sent_tokenize(label)) for label in labels]

    return preds, labels

def compute_metrics(eval_pred):
    logits, labels = eval_pred


    if isinstance(logits, tuple):
        logits = logits[0]


    # Convert to predicted token IDs by argmax over vocab dimension
    pred_ids = np.argmax(logits, axis=-1)

    # Cast labels to a NumPy array to make masking easy
    labels = np.array(labels)

    # Replace -100 with pad_token_id so that decode works correctly
    labels = np.where(labels == -100,
                      tokenizer.pad_token_id,
                      labels)

    # Decode
    decoded_preds  = tokenizer.batch_decode(
        pred_ids, skip_special_tokens=True
    )
    decoded_labels = tokenizer.batch_decode(
        labels,  skip_special_tokens=True
    )


    decoded_preds, decoded_labels = postprocess_text(
        decoded_preds, decoded_labels
    )
    result = rouge_score.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )
    # format the scores
    final_scores = {
      k: round((v.mid.fmeasure if hasattr(v, "mid") else v) * 100, 4)
      for k, v in result.items()
    }
    return final_scores


In [ ]:
# Load LoRA configuration
peft_config = LoraConfig(

    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj","lm_head" ]
)

model = get_peft_model(model, peft_config)

In [ ]:
# training parameters
training_arguments = TrainingArguments(
    auto_find_batch_size=True,
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_strategy=save_strategy,
    eval_strategy="epoch",
    logging_strategy="steps",
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    logging_steps=logging_steps,
    fp16=fp16,
    bf16=bf16,
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",    # checking for ROUGE-L F1
    greater_is_better=True,
    max_grad_norm=max_grad_norm,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard",
    overwrite_output_dir=True

)

In [ ]:
# supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    eval_dataset=eval_dataset,
    train_dataset=train_dataset,
    peft_config=peft_config,
    args=training_arguments,
    compute_metrics=compute_metrics,
    callbacks=callbacks,

)

Truncating train dataset:   0%|          | 0/144 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/36 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# Import rouge metrics
rouge_score = evaluate.load("rouge")

In [ ]:
# Train the base model
trainer.train()

In [ ]:
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)

/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:221: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


('Llama-2-7b-chat-norwegian-sum/tokenizer_config.json',
 'Llama-2-7b-chat-norwegian-sum/special_tokens_map.json',
 'Llama-2-7b-chat-norwegian-sum/tokenizer.model',
 'Llama-2-7b-chat-norwegian-sum/added_tokens.json',
 'Llama-2-7b-chat-norwegian-sum/tokenizer.json')

In [ ]:
trainer.evaluate()

In [ ]:
test_metrics = trainer.evaluate(test_dataset)
print(test_metrics)